In [31]:
import pandas as pd

In [15]:
parts = [
    "../data/split/part_1.csv",
    "../data/split/part_2.csv",
    "../data/split/part_3.csv",
    "../data/split/part_4.csv"
]
samples = []
for part in parts:
    sample = pd.read_csv(part, usecols=["owner", "name", "combined_text"])
    sample = sample.sample(frac=0.25,random_state=42)#Changing the df size to 1M beacuse of i dont have enough memory
    samples.append(sample)
df = pd.concat(samples,ignore_index=True)


In [33]:
df.shape

(1044276, 3)

In [34]:
#If i run the cosine  on this data it will be very big and impossible to do beacuse of it can jump upto TB size.

In [35]:
#creating the vector data of this df

In [36]:
import joblib
vectorizer = joblib.load('../data/models/tfidf_vectorizer.pkl')
data_matrix = vectorizer.transform(df['combined_text'])  #only this column goes into TF-IDF

In [37]:
#So we got vector data on the full dataset

In [38]:
data_matrix.nnz

11703351

In [39]:
data_matrix.shape

(1044276, 115158)

In [40]:
data_matrix.dtype

dtype('float64')

In [41]:
print(f"Memory usage: {data_matrix.data.nbytes / 1e6:.2f} MB")

Memory usage: 93.63 MB


In [42]:
#TF-IDF are higher dimensional
#TruncatedSVD reduces the dimension and make it dense so the FAISS can work efficently

In [55]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=500,random_state=42)
reduced_matrix = svd.fit_transform(data_matrix)

In [51]:
reduced_matrix.shape

(1044276, 300)

In [59]:
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.4f}") #This tells you how much information were captured into the compressed version


Explained variance: 0.3999


In [ ]:
#In here using dimension = 100 only gave as 24%. so not good

#I changed the dataset size to 1M 

#Now we are getting upto 40% is good

In [61]:
import psutil
print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.2f} GB")

Total RAM: 16.85 GB
Available RAM: 7.84 GB


In [62]:
joblib.dump(svd,"../data/models/svd_model.pkl")
joblib.dump(reduced_matrix,"../data/models/reduced_matrix.pkl")

['../data/models/reduced_matrix.pkl']

In [3]:
import joblib
reduced_matrix = joblib.load("../data/models/reduced_matrix.pkl")

In [9]:
import faiss
import numpy as np

vectors = reduced_matrix.astype(np.float32)
faiss.normalize_L2(vectors)

In [10]:
d = vectors.shape[1]
nlist = 1000

quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer,d,nlist,faiss.METRIC_INNER_PRODUCT)

In [11]:
index.train(vectors)
index.add(vectors)

In [12]:
index.nprobe = 10

In [16]:
import pandas as pd
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

In [17]:
tfidf = joblib.load("../data/models/tfidf_vectorizer.pkl")

In [19]:
svd = joblib.load("../data/models/svd_model.pkl")

In [20]:
def search_repos(query_text, top_n=5):
    #  vectorize the query using the SAME tfidf vectorizer
    query_tfidf = tfidf.transform([query_text])
    
    # reduce using the SAME fitted SVD model
    query_reduced = svd.transform(query_tfidf).astype(np.float32)

    faiss.normalize_L2(query_reduced)

    D, I = index.search(query_reduced, top_n)
    
    return df[['owner', 'name']].iloc[I[0]]

In [21]:
print(search_repos("hospital management using python or js"))

                      owner                              name
64832              lava1201          Hotel-management-using-c
362322       rahulnejanawar                  asset_management
988371              Zetolac  FortniteAntiCheatForcerUsingDate
932558   RT-Thread-packages                               LPM
1035135      StevenChiu2018           Intersection-Management


In [22]:
print(df['combined_text'].iloc[64832])

Hotel-management-using-c  C C


In [23]:
print(df['combined_text'].sample(10).tolist())

['decay-factory A simple cli to convert any image to a Decay themed wallpaper Python Python Makefile Shell bash decay factory image-factory image-processing python', 'WAV2MIDI A demo for the ResNet-18 hierarchical classification note segment system Python Python Shell', 'ionic-shop An Ionic Shopping Cart Plugin - NO LONGER SUPPORTED! JavaScript JavaScript CSS HTML', 'raster-plotter Print raster images with vector plotter Processing Processing', 'srslte-docker-emulated Minimal end-to-end LTE. Dockerized and emulated radio over shared memory. Dockerfile Dockerfile', 'fractos 3D fractal renderer written in TypeScript GLSL GLSL TypeScript JavaScript fractal fractal-rendering renderer raymarching pathtracing typescript threejs', 'asb-docker-compose This project provides tooling to run a maker on UnstoppableSwap using Docker ', 'chatgpt-java OpenAI Api Client in Java Java Java', 'tinygl  C C Shell', 'Music-Generator Procedural generation of music done in python Python Python']


In [24]:
print('hospital' in tfidf.vocabulary_)


True


In [25]:
matches = df[df['combined_text'].str.contains('hospital', case=False, na=False)]
print(matches.shape)
print(matches[['owner', 'name']].head(10))

(397, 3)
                      owner                               name
22241                alipay                         RJU_Ant_QA
23353  open-power-workgroup                           Hospital
25744                agueye                     Matlabu-Chifai
26428           small-bears                          hospitals
27608          MoH-Malaysia                     covid19-public
31481           girishsaraf  Online-Appointment-Booking-System
39713           HospitalRun                         components
42655            Trustroots                         trustroots
44531           small-bears                          hospitalq
45921        prateeksinghal         Hospital-Management-System


In [26]:
# Get TF-IDF + SVD vector for a KNOWN relevant repo
known_idx = df[df['name'] == 'Hospital-Management-System'].index[0]
known_vec = vectors[known_idx:known_idx+1]

# Compare directly to your query vector
from sklearn.metrics.pairwise import cosine_similarity
query_tfidf = tfidf.transform(["hospital management using python or js"])
query_reduced = svd.transform(query_tfidf).astype(np.float32)
faiss.normalize_L2(query_reduced)

similarity = cosine_similarity(query_reduced, known_vec)
print(similarity)

[[0.2886252]]


In [28]:
#only 40% information in the compression makes the prediction not good
#So reducing the size to 250k from 1M
